# ClinicalNotes dataset generation (v0)

Generated for Hugging Face: https://huggingface.co/datasets/jmaasch/compositional_causal_reasoning/

Version 0.

Code by Jacqueline Maasch | April 2025

In [ ]:
# General importations.
import pandas as pd
import numpy as np
import sys

sys.path.append('../compositional_causal_reasoning')

from utils import Utils
from clinical_notes import ClinicalNotes
from dataset_generator import DataSetGenerator

In [ ]:
u = Utils()
dg = DataSetGenerator()

## Step 1: Get raw dataset.

In [ ]:
# x levels of graphical complexity (captured by BCC size).
# y tasks per graphical complexity level.
# z samples per task.
# w replicates per sample.
# = x*y*z*w subtasks.
graph_sizes = [[6,4,6],[7,5,7]]
n_tasks_per_size = 1
n_samples_per_task = 1000
reps_per_sample = 5
bcc_type = "wheel"
n_extra_vars = 4

df = dg.get_dataset(task_generator = ClinicalNotes,
                    graph_sizes = graph_sizes,
                    n_tasks_per_size = n_tasks_per_size,
                    n_samples_per_task = n_samples_per_task, 
                    reps_per_sample = reps_per_sample, 
                    n_extra_vars = n_extra_vars, 
                    bcc_type = bcc_type)

print(df.info())
display(df.head(5))
display(df.tail(5))

## Step 2: Process factual and counterfactual prompts.

In [ ]:
# Process prompts.
df_factual, df_cf = dg.process_prompts()

In [ ]:
print(df_factual.info())
display(df_factual.head(5))

In [ ]:
print(df_cf.info())
display(df_cf.head(5))

In [ ]:
l = len(df_factual[(df_factual["Context ID"] == 0) & (df_factual["Effect"] == "surgery")])
print("\nTotal factual q's per quantity per task:", l)

In [ ]:
l = len(df_cf[(df_cf["Context ID"] == 0) & (df_cf["Cause-effect pair"] == ("pain", "surgery"))])
print("\nTotal counterfactual q's per quantity per task:", l)

## Step 3: Get ground truth PNS values.

Get dictionary mapping cause-effect pairs to their PNS value.

Keys are the Context ID. Values are dictionaries whose keys are the cause-effect pair and whose values are the finite sample PNS computed using ground truth response vectors.

In [ ]:
pns_dict = dg.get_pns_dict(verbose = False)
display(pns_dict)

## Step 4: Compute internal consistency thresholds.

Return a dictionary that maps compositions to their correctness threshold
for internal compositional consistency evaluation. Thresholds are the RAE
for each composition relative to the global quantity of interest, times a
multiplier of the user's choice. 

* RAE = (abs(global PNS - composition PNS) / global PNS)
* Threhold = RAE*multiplier
        
This method of obtaining the threshold accounts for the innate error owed
to PNS estimation on finite samples, while the multiplier represents the
user's tolerance level for errors larger than the finite sample error.

Keys are the Context ID. Values are dictionaries whose keys are the causal composition (denoted by a list of cause-effect pairs whose PNS values are multiplied) and whose values are the internal consistency threshold.

For public use, we export threholds with multiplier 1.0 so that the end user can select 
their own multiplier downstream.

In [ ]:
# Not for export.
threshold_dict = dg.get_internal_consistency_thresholds(multiplier = 1.25)
display(threshold_dict)

In [ ]:
# For export.
threshold_dict = dg.get_internal_consistency_thresholds(multiplier = 1.0)
display(threshold_dict)